In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

# 1) Locate latest all-sites metrics JSON
results_dir = Path("../ttm_benchmarking_results2")
all_sites_files = sorted(results_dir.glob("all_sites_metrics_*.json"), key=lambda p: p.stat().st_mtime)

if not all_sites_files:
    raise FileNotFoundError(f"No all_sites_metrics_*.json found in {results_dir.resolve()}")

latest_file = all_sites_files[-1]
print(f"Using metrics file: {latest_file}")

with latest_file.open("r") as f:
    all_sites = json.load(f)

# 2) Build one row per (site, channel)
methods = [
    ("ttm", "ttm_metrics"),
    ("mean_baseline", "mean_baseline_metrics"),
    ("median_baseline", "median_baseline_metrics"),
]
metrics = ["mse", "rmse", "mae", "mape", "r2", "smape"]

rows = []
for site_entry in sorted(all_sites, key=lambda entry: str(entry.get("site", ""))):
    site_name = site_entry.get("site")
    file_name = site_entry.get("file")

    # Channel names are consistent across all methods
    channel_names = sorted(site_entry.get("ttm_metrics", {}).get("per_channel", {}).keys())

    for channel in channel_names:
        row = {
            "site": site_name,
            "file": file_name,
            "channel": channel,
        }

        # Method-specific metric values (e.g., mse_ttm, mse_mean_baseline, ... )
        for metric in metrics:
            for method_name, method_key in methods:
                value = (
                    site_entry.get(method_key, {})
                    .get("per_channel", {})
                    .get(channel, {})
                    .get(metric, np.nan)
                )
                row[f"{metric}_{method_name}"] = value

        rows.append(row)

df_site_channel_metrics = pd.DataFrame(rows)

# 3) Order columns by metric: each with ttm / mean_baseline / median_baseline
ordered_cols = ["site", "file", "channel"]
for metric in metrics:
    ordered_cols.extend([
        f"{metric}_ttm",
        f"{metric}_mean_baseline",
        f"{metric}_median_baseline",
    ])

df_site_channel_metrics = df_site_channel_metrics[ordered_cols]
df_site_channel_metrics = df_site_channel_metrics.reset_index(drop=True)

print(f"Rows: {len(df_site_channel_metrics)}, Columns: {len(df_site_channel_metrics.columns)}")

# Optional save
output_csv = Path("site_channel_metrics_summary.csv")
df_site_channel_metrics.to_csv(output_csv, index=False)
print(f"Saved: {output_csv.resolve()}")

Using metrics file: ../ttm_benchmarking_results2/all_sites_metrics_20260222_065229.json
Rows: 828, Columns: 21
Saved: /home/rishi/ML Projects/Air Pollution/CPCB/site_channel_metrics_summary.csv


In [15]:
df_site_channel_metrics

,site,file,channel,mse_ttm,mse_mean_baseline,mse_median_baseline,rmse_ttm,rmse_mean_baseline,rmse_median_baseline,mae_ttm,...,mae_median_baseline,mape_ttm,mape_mean_baseline,mape_median_baseline,r2_ttm,r2_mean_baseline,r2_median_baseline,smape_ttm,smape_mean_baseline,smape_median_baseline
0,site_113_Shadipur_Delhi_CPCB_15Min,site_113_Shadipur_Delhi_CPCB_15Min.csv,CO (mg/m³),0.504162,0.943369,1.043430,0.710044,0.971272,1.021484,0.425137,...,0.622159,295.169434,344.514160,248.895447,0.550478,0.158872,0.069655,88.944038,127.785149,130.735733
1,site_113_Shadipur_Delhi_CPCB_15Min,site_113_Shadipur_Delhi_CPCB_15Min.csv,NO2 (µg/m³),0.613175,0.902053,0.917368,0.783055,0.949764,0.957793,0.516053,...,0.645328,898.029297,892.986084,827.282349,0.372020,0.076168,0.060483,99.549034,114.967262,116.394440
2,site_113_Shadipur_Delhi_CPCB_15Min,site_113_Shadipur_Delhi_CPCB_15Min.csv,Ozone (µg/m³),0.082692,0.120190,0.130285,0.287563,0.346684,0.360950,0.197218,...,0.260753,170.802292,216.671829,248.476578,0.385626,0.107030,0.032030,60.385418,74.325119,69.781342
3,site_113_Shadipur_Delhi_CPCB_15Min,site_113_Shadipur_Delhi_CPCB_15Min.csv,PM10 (µg/m³),0.261139,0.349084,0.377689,0.511018,0.590834,0.614564,0.336623,...,0.409384,255.091003,314.403168,288.301086,0.682027,0.574942,0.540112,58.019226,64.583557,64.266190
4,site_113_Shadipur_Delhi_CPCB_15Min,site_113_Shadipur_Delhi_CPCB_15Min.csv,PM2.5 (µg/m³),0.308590,0.377965,0.413100,0.555509,0.614789,0.642729,0.296938,...,0.366163,135.922195,138.916656,154.414536,0.445084,0.320332,0.257151,63.872292,71.581985,72.938881
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
823,site_5680_Lingaraj_Nagar_Hubballi_KSPCB_15Min,site_5680_Lingaraj_Nagar_Hubballi_KSPCB_15Min.csv,NO2 (µg/m³),0.211653,0.225298,0.287798,0.460058,0.474656,0.536468,0.356584,...,0.352090,301.677094,182.917114,374.242096,0.119806,0.063061,-0.196855,94.058258,97.832062,93.827065
824,site_5680_Lingaraj_Nagar_Hubballi_KSPCB_15Min,site_5680_Lingaraj_Nagar_Hubballi_KSPCB_15Min.csv,Ozone (µg/m³),0.027748,0.034791,0.039282,0.166578,0.186523,0.198198,0.123226,...,0.153513,941.974182,1055.240112,1082.415161,0.443663,0.302460,0.212408,23.453615,26.006245,27.157185
825,site_5680_Lingaraj_Nagar_Hubballi_KSPCB_15Min,site_5680_Lingaraj_Nagar_Hubballi_KSPCB_15Min.csv,PM10 (µg/m³),0.012595,0.015528,0.015290,0.112227,0.124611,0.123654,0.068149,...,0.083820,16.701248,20.011095,19.550724,0.666916,0.589349,0.595628,13.358402,15.924681,15.822892
826,site_5680_Lingaraj_Nagar_Hubballi_KSPCB_15Min,site_5680_Lingaraj_Nagar_Hubballi_KSPCB_15Min.csv,PM2.5 (µg/m³),0.027065,0.032625,0.033184,0.164513,0.180625,0.182165,0.103967,...,0.126914,207.285721,221.240875,213.839096,0.496609,0.393181,0.382786,65.842178,73.566994,73.334145


In [17]:
# Metric summary per channel (mean and median across all sites)
metric_cols = [c for c in df_site_channel_metrics.columns if c not in ("site", "file", "channel")]

summary_per_channel = (
    df_site_channel_metrics
    .groupby("channel")[metric_cols]
    .agg(["mean", "median"])
)

# Flatten MultiIndex columns: (metric_method, stat) -> metric_method_stat
summary_per_channel.columns = [f"{col}_{stat}" for col, stat in summary_per_channel.columns]

# Re-order so columns are grouped by metric: metric_ttm_mean, metric_ttm_median, ...
ordered_summary_cols = []
for metric in metrics:
    for method_name, _ in methods:
        ordered_summary_cols.extend([
            f"{metric}_{method_name}_mean",
            f"{metric}_{method_name}_median",
        ])

ordered_summary_cols = [c for c in ordered_summary_cols if c in summary_per_channel.columns]
summary_per_channel = summary_per_channel[ordered_summary_cols]

summary_per_channel

,mse_ttm_mean,mse_ttm_median,mse_mean_baseline_mean,mse_mean_baseline_median,mse_median_baseline_mean,mse_median_baseline_median,rmse_ttm_mean,rmse_ttm_median,rmse_mean_baseline_mean,rmse_mean_baseline_median,...,r2_mean_baseline_mean,r2_mean_baseline_median,r2_median_baseline_mean,r2_median_baseline_median,smape_ttm_mean,smape_ttm_median,smape_mean_baseline_mean,smape_mean_baseline_median,smape_median_baseline_mean,smape_median_baseline_median
channel,,,,,,,,,,,,,,,,,,,,,
CO (mg/m³),0.606280,0.405235,0.807934,0.599096,0.853683,0.647373,0.685735,0.636577,0.808002,0.774013,...,0.200729,0.155335,0.152081,0.108900,78.080023,78.181572,91.942260,91.507938,88.779210,88.682064
NO2 (µg/m³),0.459817,0.266789,0.694469,0.432256,0.745995,0.451919,0.549542,0.516516,0.679845,0.657461,...,0.239673,0.239134,0.195592,0.178795,56.777345,58.977322,70.827045,73.859901,67.290814,72.011497
Ozone (µg/m³),0.531227,0.280193,0.912105,0.551074,0.976704,0.578192,0.604114,0.529333,0.816264,0.742344,...,0.029810,0.040958,-0.002445,-0.021186,66.927389,66.224907,98.795039,104.640862,85.420915,88.812614
PM10 (µg/m³),0.596193,0.432743,0.755085,0.580755,0.792419,0.618351,0.715862,0.657830,0.813115,0.762072,...,0.307340,0.335608,0.275716,0.293560,70.773905,70.307575,82.397509,82.569763,79.639236,82.186619
PM2.5 (µg/m³),0.635029,0.485069,0.781499,0.669610,0.822141,0.699973,0.735328,0.696469,0.823669,0.818296,...,0.317305,0.317098,0.283547,0.283077,67.444268,63.722044,77.152342,72.994461,75.070220,71.751194
SO2 (µg/m³),0.915150,0.446523,1.162677,0.561631,1.224072,0.586117,0.798219,0.668223,0.899247,0.749414,...,0.112884,0.065884,0.078170,0.031934,76.113470,79.734367,87.690180,91.106663,83.585426,86.092770
